In [1]:
import sys
sys.path.append('..')  # so we can import from models/

from models.bsm import bsm_price, bsm_greeks, bsm_greeks_fd
import numpy as np

In [2]:
# Benchmark 1: Single option (canonical case)
price = bsm_price(S=100, K=100, T=1.0, r=0.05, sigma=0.20, option_type='call')
print(f"Single ATM call: {price}")

# Benchmark 2: Vector of strikes
strikes = np.array([80, 90, 100, 110, 120])
prices = bsm_price(S=100, K=strikes, T=1.0, r=0.05, sigma=0.20, option_type='call')
print(f"Vector of strikes: {prices}")
print(f"Shape: {prices.shape}")

# Benchmark 3: 2D grid (the real test of broadcasting)
strikes = np.array([90, 100, 110])         # shape (3,)
maturities = np.array([[0.25], [0.5], [1.0], [2.0]])  # shape (4, 1)
grid = bsm_price(S=100, K=strikes, T=maturities, r=0.05, sigma=0.20, option_type='call')
print(f"\nGrid shape: {grid.shape}")
print(f"Grid:\n{grid}")

Single ATM call: 10.450583572185565
Vector of strikes: [24.58883544 16.69944841 10.45058357  6.04008813  3.24747742]
Shape: (5,)

Grid shape: (4, 3)
Grid:
[[11.67008669  4.61499713  1.19113166]
 [13.49851748  6.88872858  2.90647132]
 [16.69944841 10.45058357  6.04008813]
 [22.03338001 16.12677972 11.45545587]]


In [3]:
# Benchmark 4: Put-Call Parity 2D grid 
C = bsm_price(S=100, K=strikes, T=maturities, r=0.05, sigma=0.20, option_type='call')
P = bsm_price(S=100, K=strikes, T=maturities, r=0.05, sigma=0.20, option_type='put')
parity_residual = C - P - 100 + strikes * np.exp(-0.05 * maturities)
MAR = np.max(np.abs(parity_residual))
print(f"\nGrid shape: {parity_residual.shape}")
print(f"Grid:\n{parity_residual}")
print(f"\nMax Absolute Residual: {MAR}")
assert MAR <= 1e-10



Grid shape: (4, 3)
Grid:
[[ 0.00000000e+00  0.00000000e+00  1.42108547e-14]
 [-1.42108547e-14  0.00000000e+00  1.42108547e-14]
 [ 0.00000000e+00  0.00000000e+00 -1.42108547e-14]
 [ 1.42108547e-14  1.42108547e-14  0.00000000e+00]]

Max Absolute Residual: 1.4210854715202004e-14


In [4]:
# Greek Test 
# Example:
S = 100
K = 100
T = 1
r = 0.05
sigma = 0.2

call_price = bsm_price(S, K, T, r, sigma, option_type='call')
put_price = bsm_price(S, K, T, r, sigma, option_type='put')

print(f"Call Price: {call_price:.4f}")
print(f"Put Price: {put_price:.4f}")

call_greeks = bsm_greeks(S, K, T, r, sigma, option_type='call')
put_greeks = bsm_greeks(S, K, T, r, sigma, option_type='put')

print("Call Greeks:", call_greeks)
print("Put Greeks:", put_greeks)

# Call PDE test 
print("Call PDE Test:")
print(call_greeks['theta'] + 0.5 * sigma**2 * S**2 * call_greeks['gamma'] + r * S * call_greeks['delta'] - r * call_price)

Call Price: 10.4506
Put Price: 5.5735
Call Greeks: {'delta': np.float64(0.6368306511756191), 'gamma': np.float64(0.018762017345846895), 'vega': np.float64(37.52403469169379), 'theta': np.float64(-6.414027546438197), 'rho': np.float64(53.232481545376345)}
Put Greeks: {'delta': np.float64(-0.3631693488243809), 'gamma': np.float64(0.018762017345846895), 'vega': np.float64(37.52403469169379), 'theta': np.float64(-1.657880423934626), 'rho': np.float64(-41.89046090469506)}
Call PDE Test:
-2.220446049250313e-16


In [5]:
# Greek Test with Finite Difference Comparison
# Example:
S = 100
K = 100
T = 1
r = 0.05
sigma = 0.2

call_greeks_fd = bsm_greeks_fd(S, K, T, r, sigma, option_type='call', h = 1e-4)
put_greeks_fd = bsm_greeks_fd(S, K, T, r, sigma, option_type='put', h = 1e-4)
call_diff = {k: abs(call_greeks_fd[k] - call_greeks[k]) for k in call_greeks}
put_diff = {k: abs(put_greeks_fd[k] - put_greeks[k]) for k in put_greeks}

print("Call Greeks:", call_greeks)
print("Put Greeks:", put_greeks)
print("Call Greeks FD:", call_greeks_fd)
print("Put Greeks FD:", put_greeks_fd)
print("Abs Diff - Call:", call_diff)
print("Abs Diff - Put:", put_diff)



Call Greeks: {'delta': np.float64(0.6368306511756191), 'gamma': np.float64(0.018762017345846895), 'vega': np.float64(37.52403469169379), 'theta': np.float64(-6.414027546438197), 'rho': np.float64(53.232481545376345)}
Put Greeks: {'delta': np.float64(-0.3631693488243809), 'gamma': np.float64(0.018762017345846895), 'vega': np.float64(37.52403469169379), 'theta': np.float64(-1.657880423934626), 'rho': np.float64(-41.89046090469506)}
Call Greeks FD: {'delta': np.float64(0.6368306511816968), 'gamma': np.float64(0.018761880937745445), 'vega': np.float64(37.52403438721075), 'theta': np.float64(-6.414027551358004), 'rho': np.float64(53.23248077413467)}
Put Greeks FD: {'delta': np.float64(-0.36316934885149976), 'gamma': np.float64(0.018761880937745445), 'vega': np.float64(37.52403438721075), 'theta': np.float64(-1.6578804288513993), 'rho': np.float64(-41.89046183441292)}
Abs Diff - Call: {'delta': np.float64(6.0776939037054944e-12), 'gamma': np.float64(1.3640810144974203e-07), 'vega': np.float6